# **Notebook 05 — OTel 2.0 LLM-Only Evaluation**

**Module:** 2 — Expanded Multi-Model Evaluation  
**System:** `farbodtavakkoli/OTel-2.0-LLM-31B-IT` without RAG  
**Purpose:** Evaluate the newer telecom-specialized OTel 2.0 model on the same frozen Track 1 and Track 2 benchmark suites.


# **1. Environment Initialisation**

## **1.1. Load Dependencies**

### **Install All Required Libraries**

In [1]:
!pip install opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp

INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 8.5 MB/s eta 0:00:00


In [2]:
# =============================================================================
# OTEL 2.0 LLM-ONLY SETUP
# INSTALL DEPENDENCIES
# =============================================================================

# Install the libraries required for telecom RAG data acquisition,
# document processing, embeddings and vector retrieval.
#
# The current runtime is CPU-based because inference is not yet being
# performed. GPU-specific acceleration can be enabled later when required.

!pip install -q \
    transformers \
    accelerate \
    bitsandbytes \
    huggingface_hub \
    safetensors \
    sentencepiece \
    sentence-transformers \
    faiss-cpu \
    pypdf \
    python-docx \
    python-pptx \
    beautifulsoup4 \
    pyarrow \
    tqdm

print("All required Module 2 libraries installed successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 117.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 20.5 MB/s eta 0:00:00
All required Module 2 libraries installed successfully.


### **Import All Required Libraries**

In [3]:
# =============================================================================
# OTEL 2.0 LLM-ONLY SETUP
# IMPORT REQUIRED LIBRARIES
# =============================================================================

# ---------------------------------------------------------------------------
# Core scientific stack
# ---------------------------------------------------------------------------
import numpy as np
import scipy
import pandas as pd

# ---------------------------------------------------------------------------
# Standard Python libraries
# ---------------------------------------------------------------------------
import os
import gc
import json
import shutil
import time
from pathlib import Path

# ---------------------------------------------------------------------------
# Progress monitoring
# ---------------------------------------------------------------------------
from tqdm.auto import tqdm

# Data / document processing
# ---------------------------------------------------------------------------
import pyarrow
import pyarrow.parquet as pq
from docx import Document
from bs4 import BeautifulSoup
from pypdf import PdfReader

# ---------------------------------------------------------------------------
# HTTP / source acquisition
# ---------------------------------------------------------------------------
import requests

# ---------------------------------------------------------------------------
# PyTorch
# ---------------------------------------------------------------------------
import torch

# ---------------------------------------------------------------------------
# Hugging Face / LLM inference
# ---------------------------------------------------------------------------
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline,
)

from huggingface_hub import login, HfApi, snapshot_download

# ---------------------------------------------------------------------------
# Embeddings
# ---------------------------------------------------------------------------
from sentence_transformers import SentenceTransformer

# ---------------------------------------------------------------------------
# Vector similarity search
# ---------------------------------------------------------------------------
import faiss

# ---------------------------------------------------------------------------
# PDF document processing
# ---------------------------------------------------------------------------
from pypdf import PdfReader

print("All required libraries imported successfully.")

# Display key package versions for reproducibility
print("\nPackage Versions")
print("-" * 40)
print(f"NumPy           : {np.__version__}")
print(f"SciPy           : {scipy.__version__}")
print(f"Pandas          : {pd.__version__}")
print(f"PyTorch         : {torch.__version__}")
print(f"FAISS           : {faiss.__version__}")
print(f"PyArrow         : {pyarrow.__version__}")

All required libraries imported successfully.

Package Versions
----------------------------------------
NumPy           : 2.1.3
SciPy           : 1.16.3
Pandas          : 2.2.3
PyTorch         : 2.11.0+cu128
FAISS           : 1.15.0
PyArrow         : 18.1.0


### **Import Datasets from Kaggle**

In [4]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [5]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

cliffordimaguezegie_telecom_benchmark_path = kagglehub.dataset_download('cliffordimaguezegie/benchmark')

print('Data source import complete.')


100%|██████████| 21.9k/21.9k [00:00<00:00, 31.8MB/s]

Extracting files...
Data source import complete.


In [6]:
print("QUESTIONS:")
print(cliffordimaguezegie_telecom_benchmark_path)

QUESTIONS:
/root/.cache/kagglehub/datasets/cliffordimaguezegie/benchmark/versions/1


In [7]:
from pathlib import Path


BENCHMARK_DIR = Path(
    cliffordimaguezegie_telecom_benchmark_path
)

print("BENCHMARK :", BENCHMARK_DIR)

BENCHMARK : /root/.cache/kagglehub/datasets/cliffordimaguezegie/benchmark/versions/1


In [8]:
# =============================================================================
# RAG V1 — INSPECT BENCHMARK DATASET
# =============================================================================

print("=" * 90)
print("RAG V1 — BENCHMARK DATASET CONTENTS")
print("=" * 90)

for file_path in sorted(
    BENCHMARK_DIR.rglob("*")
):

    if file_path.is_file():

        print(
            file_path.relative_to(
                BENCHMARK_DIR
            )
        )

print("=" * 90)

RAG V1 — BENCHMARK DATASET CONTENTS
track1_20_questions.json
track2_final_32_questions.json


In [9]:
# =============================================================================
# LOAD BENCHMARK QUESTION BANKS
# =============================================================================

import json


TRACK1_FILE = (
    BENCHMARK_DIR
    / "track1_20_questions.json"
)

TRACK2_FILE = (
    BENCHMARK_DIR
    / "track2_final_32_questions.json"
)


# =============================================================================
# LOAD TRACK 1
# =============================================================================

with open(
    TRACK1_FILE,
    "r",
    encoding="utf-8",
) as file:

    benchmark_questions = json.load(
        file
    )


# =============================================================================
# LOAD TRACK 2
# =============================================================================

with open(
    TRACK2_FILE,
    "r",
    encoding="utf-8",
) as file:

    benchmark_questions_track2 = json.load(
        file
    )


# =============================================================================
# VALIDATION
# =============================================================================

print("=" * 90)
print("RAG V1 — BENCHMARK QUESTION BANKS LOADED")
print("=" * 90)

print("\nTRACK 1")
print("-" * 60)

print(
    f"Questions : "
    f"{len(benchmark_questions)}"
)

print(
    f"First ID  : "
    f"{benchmark_questions[0]['id']}"
)

print(
    f"Last ID   : "
    f"{benchmark_questions[-1]['id']}"
)


print("\nTRACK 2")
print("-" * 60)

print(
    f"Questions : "
    f"{len(benchmark_questions_track2)}"
)

print(
    f"First ID  : "
    f"{benchmark_questions_track2[0]['evaluation_id']}"
)

print(
    f"Last ID   : "
    f"{benchmark_questions_track2[-1]['evaluation_id']}"
)


# =============================================================================
# COUNT CHECKS
# =============================================================================

if len(benchmark_questions) != 20:

    raise RuntimeError(
        f"Track 1 expected 20 questions, "
        f"found {len(benchmark_questions)}."
    )


if len(benchmark_questions_track2) != 32:

    raise RuntimeError(
        f"Track 2 expected 32 questions, "
        f"found {len(benchmark_questions_track2)}."
    )


print("\n" + "=" * 90)
print("TRACK 1 + TRACK 2 QUESTION BANKS VALIDATED")
print("=" * 90)

RAG V1 — BENCHMARK QUESTION BANKS LOADED

TRACK 1
------------------------------------------------------------
Questions : 20
First ID  : Q01
Last ID   : Q20

TRACK 2
------------------------------------------------------------
Questions : 32
First ID  : T2-01
Last ID   : T2-32

TRACK 1 + TRACK 2 QUESTION BANKS VALIDATED


### **GPU Verification**

In [10]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# GPU / CUDA VERIFICATION
# =============================================================================

import torch


print("=" * 80)
print("GPU / CUDA VERIFICATION")
print("=" * 80)

print(f"PyTorch version   : {torch.__version__}")
print(f"CUDA available    : {torch.cuda.is_available()}")

if torch.cuda.is_available():

    print(f"CUDA version      : {torch.version.cuda}")
    print(f"GPU count         : {torch.cuda.device_count()}")

    for i in range(torch.cuda.device_count()):

        props = torch.cuda.get_device_properties(i)

        total_memory = (
            props.total_memory / (1024**3)
        )

        allocated = (
            torch.cuda.memory_allocated(i)
            / (1024**3)
        )

        reserved = (
            torch.cuda.memory_reserved(i)
            / (1024**3)
        )

        free_memory = max(
            0,
            total_memory - reserved
        )

        print(f"\nGPU {i}")
        print(f"  Name           : {props.name}")
        print(
            f"  Total Memory   : "
            f"{total_memory:.2f} GB"
        )
        print(
            f"  Free Memory*   : "
            f"{free_memory:.2f} GB"
        )
        print(
            f"  Allocated      : "
            f"{allocated:.2f} GB"
        )
        print(
            f"  Reserved       : "
            f"{reserved:.2f} GB"
        )
        print(
            f"  Compute Cap.   : "
            f"{props.major}.{props.minor}"
        )

    # -------------------------------------------------------------------------
    # Select first available GPU
    # -------------------------------------------------------------------------

    DEVICE = torch.device("cuda:0")

    torch.cuda.set_device(0)

    print(f"\nSelected device  : {DEVICE}")

else:

    DEVICE = torch.device("cpu")

    print("\nWARNING: CUDA GPU not detected.")
    print(
        "In Colab, enable GPU via "
        "Runtime → Change runtime type → T4 GPU "
        "(or the GPU available to your runtime)."
    )

print("\nGPU verification completed.")

GPU / CUDA VERIFICATION
PyTorch version   : 2.11.0+cu128
CUDA available    : True
CUDA version      : 12.8
GPU count         : 1

GPU 0
  Name           : NVIDIA A100-SXM4-80GB
  Total Memory   : 79.25 GB
  Free Memory*   : 79.25 GB
  Allocated      : 0.00 GB
  Reserved       : 0.00 GB
  Compute Cap.   : 8.0

Selected device  : cuda:0

GPU verification completed.


## **1.2. Load OTel 2.0 LLM**

In [24]:
import sys
import gc
import time
import torch

def force_unload_model_and_flush(model_var_name="otel_model", tokenizer_var_name="otel_tokenizer"):
    """
    Deletes the model and tokenizer objects from Python memory,
    runs garbage collection, and flushes the CUDA cache.
    """
    print("=" * 90)
    print("STARTING COMPLETE VRAM & RAM FLUSH")
    print("=" * 90)

    # 1. Delete explicit global references if they exist
    globals_dict = globals()

    if model_var_name in globals_dict:
        print(f"--> Deleting '{model_var_name}' object from global scope...")
        del globals_dict[model_var_name]

    if tokenizer_var_name in globals_dict:
        print(f"--> Deleting '{tokenizer_var_name}' object from global scope...")
        del globals_dict[tokenizer_var_name]

    # Also check and clean standard fallback names just in case
    for fallback in ["general_model", "general_tokenizer", "model", "tokenizer"]:
        if fallback in globals_dict:
            print(f"--> Deleting fallback '{fallback}' object...")
            del globals_dict[fallback]

    # 2. Force Python Garbage Collection
    print("--> Running Python Garbage Collector (GC)...")
    gc.collect()

    # 3. Synchronize & Flush CUDA VRAM
    if torch.cuda.is_available():
        print("--> Emptying PyTorch CUDA Cache & IPC memory...")
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.synchronize()  # Block execution until release finishes

        vram_allocated = torch.cuda.memory_allocated(0) / (1024**3)
        vram_reserved = torch.cuda.memory_reserved(0) / (1024**3)
        print(f"\n[Post-Flush GPU Status]")
        print(f"VRAM Allocated : {vram_allocated:.2f} GB")
        print(f"VRAM Reserved  : {vram_reserved:.2f} GB")
    else:
        print("--> No CUDA GPU detected.")

    time.sleep(1.0)
    print("=" * 90)
    print("MEMORIES FLUSHED - READY TO LOAD NEW MODEL")
    print("=" * 90)

# Run the flush function
force_unload_model_and_flush()

STARTING COMPLETE VRAM & RAM FLUSH
--> Running Python Garbage Collector (GC)...
--> Emptying PyTorch CUDA Cache & IPC memory...

[Post-Flush GPU Status]
VRAM Allocated : 32.75 GB
VRAM Reserved  : 68.26 GB
MEMORIES FLUSHED - READY TO LOAD NEW MODEL


In [11]:
# =============================================================================
# LOAD OTEL-2.0-LLM
# =============================================================================

import time
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)


# =============================================================================
# OTEL LLM CONFIGURATION
# =============================================================================

OTEL_LLM_NAME = (
    "farbodtavakkoli/OTel-2.0-LLM-31B-IT"
)

OTEL_TEMPERATURE = 0.0
OTEL_TOP_K = 50
OTEL_TOP_P = 0.95

OTEL_MAX_NEW_TOKENS = 512


# =============================================================================
# GPU STATUS
# =============================================================================

print("=" * 90)
print("OTEL-2.0 LLM")
print("=" * 90)

print(
    f"GPU             : "
    f"{torch.cuda.get_device_name(0)}"
)

print(
    f"GPU memory      : "
    f"{torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB"
)


# =============================================================================
# LOAD TOKENIZER
# =============================================================================

print("\nLoading OTEL-2.0 LLM tokenizer...")

tokenizer_start = time.time()

otel_tokenizer = (
    AutoTokenizer.from_pretrained(
        OTEL_LLM_NAME,
        trust_remote_code=True,
    )
)

tokenizer_elapsed = (
    time.time()
    - tokenizer_start
)


# =============================================================================
# LOAD MODEL
# =============================================================================

print(
    "\nLoading OTEL-2.0 LLM model..."
)

model_start = time.time()

otel_model = (
    AutoModelForCausalLM.from_pretrained(
        OTEL_LLM_NAME,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
)

otel_model.eval()

model_elapsed = (
    time.time()
    - model_start
)


# =============================================================================
# VALIDATION
# =============================================================================

print("\nConfiguration")
print("-" * 60)

print(
    f"Model            : "
    f"{OTEL_LLM_NAME}"
)

print(
    f"Quantization     : None (Full Precision)"
)

print(
    f"Compute dtype    : bfloat16"
)

print(
    f"Temperature      : "
    f"{OTEL_TEMPERATURE}"
)

print(
    f"Top-k            : "
    f"{OTEL_TOP_K}"
)

print(
    f"Top-p            : "
    f"{OTEL_TOP_P}"
)

print(
    f"Max new tokens   : "
    f"{OTEL_MAX_NEW_TOKENS}"
)

print(
    f"Tokenizer time   : "
    f"{tokenizer_elapsed:.2f} sec"
)

print(
    f"Model load time  : "
    f"{model_elapsed:.2f} sec"
)

print(
    f"\nDevice map       : "
    f"{getattr(otel_model, 'hf_device_map', 'N/A')}"
)


# =============================================================================
# FINAL STATUS
# =============================================================================

print("\n" + "=" * 90)
print("OTEL-2.0 LLM LOADED SUCCESSFULLY")
print("=" * 90)

OTEL-2.0 LLM
GPU             : NVIDIA A100-SXM4-80GB
GPU memory      : 79.25 GB

Loading OTEL-2.0 LLM tokenizer...


config.json:   0%|          | 0.00/2.80k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.70k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/833 [00:00<?, ?it/s]

[transformers] Gemma4ForCausalLM LOAD REPORT from: farbodtavakkoli/OTel-2.0-LLM-31B-IT
Key                                                  | Status     |  | 
-----------------------------------------------------+------------+--+-
model.embed_tokens.embed_scale                       | UNEXPECTED |  | 
model.rotary_emb.full_attention_original_inv_freq    | UNEXPECTED |  | 
model.rotary_emb.sliding_attention_original_inv_freq | UNEXPECTED |  | 
model.rotary_emb.full_attention_inv_freq             | UNEXPECTED |  | 
model.rotary_emb.sliding_attention_inv_freq          | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Configuration
------------------------------------------------------------
Model            : farbodtavakkoli/OTel-2.0-LLM-31B-IT
Quantization     : None (Full Precision)
Compute dtype    : bfloat16
Temperature      : 0.0
Top-k            : 50
Top-p            : 0.95
Max new tokens   : 512
Tokenizer time   : 5.76 sec
Model load time  : 188.97 sec

Device map       : N/A

OTEL-2.0 LLM LOADED SUCCESSFULLY


## **1.3. Post GPU Verification**

In [12]:
# =============================================================================
# CPU / GPU MEMORY STATUS
# =============================================================================

import os
import psutil
import torch


# =============================================================================
# CPU / RAM
# =============================================================================

process = psutil.Process(os.getpid())

system_ram = psutil.virtual_memory()

process_ram_gb = (
    process.memory_info().rss / (1024**3)
)

total_ram_gb = (
    system_ram.total / (1024**3)
)

available_ram_gb = (
    system_ram.available / (1024**3)
)

used_ram_gb = (
    system_ram.used / (1024**3)
)


print("=" * 90)
print("RAG V1 — CPU / GPU MEMORY STATUS")
print("=" * 90)

print("\nCPU / SYSTEM RAM")
print("-" * 40)

print(
    f"Total RAM       : {total_ram_gb:.2f} GB"
)

print(
    f"Used RAM        : {used_ram_gb:.2f} GB"
)

print(
    f"Available RAM   : {available_ram_gb:.2f} GB"
)

print(
    f"Python process   : {process_ram_gb:.2f} GB"
)


# =============================================================================
# GPU
# =============================================================================

print("\nGPU / VRAM")
print("-" * 40)

print(
    f"GPU count       : "
    f"{torch.cuda.device_count()}"
)

for gpu_id in range(
    torch.cuda.device_count()
):

    props = torch.cuda.get_device_properties(
        gpu_id
    )

    allocated_gb = (
        torch.cuda.memory_allocated(gpu_id)
        / (1024**3)
    )

    reserved_gb = (
        torch.cuda.memory_reserved(gpu_id)
        / (1024**3)
    )

    total_gpu_gb = (
        props.total_memory
        / (1024**3)
    )

    free_gpu_gb = (
        total_gpu_gb
        - reserved_gb
    )

    print(
        f"\nGPU {gpu_id}: "
        f"{props.name}"
    )

    print(
        f"  Total VRAM     : "
        f"{total_gpu_gb:.2f} GB"
    )

    print(
        f"  Allocated VRAM : "
        f"{allocated_gb:.2f} GB"
    )

    print(
        f"  Reserved VRAM  : "
        f"{reserved_gb:.2f} GB"
    )

    print(
        f"  Approx. free   : "
        f"{free_gpu_gb:.2f} GB"
    )


print("\n" + "=" * 90)

RAG V1 — CPU / GPU MEMORY STATUS

CPU / SYSTEM RAM
----------------------------------------
Total RAM       : 167.05 GB
Used RAM        : 6.18 GB
Available RAM   : 159.33 GB
Python process   : 2.07 GB

GPU / VRAM
----------------------------------------
GPU count       : 1

GPU 0: NVIDIA A100-SXM4-80GB
  Total VRAM     : 79.25 GB
  Allocated VRAM : 57.18 GB
  Reserved VRAM  : 61.14 GB
  Approx. free   : 18.11 GB



# **2. Inference Pipeline**

## **2.1. Response Function**

In [13]:
import torch

def generate_response(
    model,
    tokenizer,
    question,
    system_prompt=None,
    max_new_tokens=500,
    do_sample=False,
    temperature=0.01,
    top_p=0.95,
    top_k=50,
    repetition_penalty=1.1,
):
    """Generate a standardised response from a loaded LLM."""

    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    messages.append({"role": "user", "content": question})

    # Use the model's native chat template when available.
    if getattr(tokenizer, "chat_template", None):
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        prompt = question

    # Tokenise the formatted prompt.
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    )

    # Place inputs on the model's execution device.
    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    # Define generation parameters explicitly.
    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "repetition_penalty": repetition_penalty,
        "pad_token_id": tokenizer.pad_token_id or tokenizer.eos_token_id,
    }

    if do_sample:
        generation_kwargs.update({
            "temperature": temperature,
            "top_p": top_p,
            "top_k": top_k,
        })

    # Generate response without gradient calculation.
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            **generation_kwargs,
        )

    # Decode only newly generated tokens.
    input_length = inputs["input_ids"].shape[-1]

    response = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True,
    )

    return response.strip()

## **2.2. Test Response Function**

In [14]:
print("=" * 80)
print("TEST RUN: GENERAL LLM RESPONSE")
print("=" * 80)

general_response = generate_response(
    model=otel_model,
    tokenizer=otel_tokenizer,
    question="What is the role of the AMF in a 5G Standalone network?",
    system_prompt="You are an expert telecommunications network engineer."
)

print(general_response)

TEST RUN: GENERAL LLM RESPONSE
In a 5G Standalone (SA) network, the **AMF (Access and Mobility Management Function)** serves as the primary control-plane anchor for user equipment (UE). It acts as the central point of contact between the UE and the rest of the core network during initial access and mobility events.

The AMF’s responsibilities can be categorized into four major functional areas:

### 1. Registration Management
The AMF handles all aspects of UE registration to the network:
*   **Initial Registration:** Manages the procedure when a device first connects to the 5G system.
*   **Mobility Registration Updates:** Handles updates when the UE moves across tracking areas or changes its connection state.
*   **Deregistration:** Processes requests when a UE leaves the network or loses connectivity.
*   **NAS Signaling Termination:** Acts as the termination point for Non-Access Stratum (NAS) signaling messages exchanged with the UE.

### 2. Connection Management
The AMF manages the

# **3. Telecom Benchmark Evaluation**

## **OTel 2.0 LLM-Only Inference**

### **OTel 2.0 Generation Configuration**

In [15]:
# =============================================================================
# GENERAL LLM GENERATION CONFIGURATION
# =============================================================================

GENERAL_GENERATION_CONFIG = {
    "temperature": 0.01,
    "top_k": 50,
    "top_p": 0.95,
    "repetition_penalty": 1.05,
    "max_new_tokens": 1024,
    "do_sample": True,
}


print("=" * 90)
print("GENERAL LLM GENERATION CONFIGURATION")
print("=" * 90)

print(
    f"Temperature        : "
    f"{GENERAL_GENERATION_CONFIG['temperature']}"
)

print(
    f"Top-k              : "
    f"{GENERAL_GENERATION_CONFIG['top_k']}"
)

print(
    f"Top-p              : "
    f"{GENERAL_GENERATION_CONFIG['top_p']}"
)

print(
    f"Repetition penalty : "
    f"{GENERAL_GENERATION_CONFIG['repetition_penalty']}"
)

print(
    f"Max new tokens     : "
    f"{GENERAL_GENERATION_CONFIG['max_new_tokens']}"
)

print(
    f"Do sample          : "
    f"{GENERAL_GENERATION_CONFIG['do_sample']}"
)

print("=" * 90)

GENERAL LLM GENERATION CONFIGURATION
Temperature        : 0.01
Top-k              : 50
Top-p              : 0.95
Repetition penalty : 1.05
Max new tokens     : 1024
Do sample          : True


In [19]:
# Force-reset OpenTelemetry global state in the active kernel
from opentelemetry import trace, metrics
trace._TRACER_PROVIDER = None
metrics._METER_PROVIDER = None

### **Track 1 Inference**

In [16]:
import gc
import json
import time
from datetime import datetime, timezone
import torch

# =============================================================================
# HELPER: AGGRESSIVE HARDWARE SYNCHRONIZED MEMORY CLEARING
# =============================================================================

def flush_vram(delay_sec: float = 0.5):
    """
    Forces garbage collection, releases cached CUDA VRAM back to the GPU,
    synchronizes CPU/GPU threads, and introduces a brief settlement delay.
    """
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.synchronize()  # Block CPU until GPU cache release finishes
    if delay_sec > 0:
        time.sleep(delay_sec)


# =============================================================================
# CONFIGURATION & METADATA SETUP
# =============================================================================

SYSTEM_PROMPT = "You are an expert telecommunications network engineer."

# Resolve target model & tokenizer safely (supporting otel variables)
active_model = globals().get("otel_model", globals().get("general_model"))
active_tokenizer = globals().get("otel_tokenizer", globals().get("general_tokenizer"))

if active_model is None or active_tokenizer is None:
    raise NameError("Neither 'otel_model'/'otel_tokenizer' nor 'general_model'/'general_tokenizer' was found in memory.")

MODEL_NAME = getattr(active_model.config, "_name_or_path", "farbodtavakkoli/OTel-2.0-LLM-31B-IT")
TIMESTAMP_STR = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
OUTPUT_FILE = f"track1_general_llm_results_{TIMESTAMP_STR}.json"

# Ensure benchmark dataset exists
if "benchmark_questions" not in globals():
    raise NameError("Variable 'benchmark_questions' is missing. Load your benchmark dataset first.")

track1_general_results = []
track1_start = time.time()

# Initial hardware memory flush
flush_vram(delay_sec=1.0)


# =============================================================================
# HEADER
# =============================================================================

print("=" * 90)
print("BASELINE — TRACK 1 | GENERAL LLM ONLY (NO RAG)")
print("DIRECT INFERENCE")
print("=" * 90)
print(f"Model ID        : {MODEL_NAME}")
print(f"Questions       : {len(benchmark_questions)}")
print("=" * 90)


# =============================================================================
# EXECUTE ALL QUESTIONS
# =============================================================================

for index, item in enumerate(benchmark_questions, start=1):

    # 1. PRE-QUERY CLEANUP & SYNCHRONIZATION
    flush_vram(delay_sec=0.5)

    question_id = item["id"]
    category = item["category"]
    question = item["question"]

    print(f"\n[{index:02d}/{len(benchmark_questions):02d}] {question_id} | {category}")
    start_time = time.time()

    # =========================================================================
    # INFERENCE (wrapped in torch.inference_mode to disable autograd tracking)
    # =========================================================================
    try:
        with torch.inference_mode():
            # Generate standalone LLM response using the active model/tokenizer
            answer = generate_response(
                model=active_model,
                tokenizer=active_tokenizer,
                question=question,
                system_prompt=SYSTEM_PROMPT,
                max_new_tokens=GENERAL_GENERATION_CONFIG.get("max_new_tokens", 500),
                do_sample=GENERAL_GENERATION_CONFIG.get("do_sample", False),
                temperature=GENERAL_GENERATION_CONFIG.get("temperature", 0.01),
                repetition_penalty=GENERAL_GENERATION_CONFIG.get("repetition_penalty", 1.1),
            )

            # Compute input token length using active tokenizer
            prompt_str = f"{SYSTEM_PROMPT}\n{question}"
            input_tokens = len(active_tokenizer.encode(prompt_str))
            output_tokens = len(active_tokenizer.encode(answer))

        elapsed_sec = time.time() - start_time

        record = {
            "question_id": question_id,
            "category": category,
            "question": question,
            "expected_points": item.get("expected_points", []),
            "model_id": MODEL_NAME,
            "mode": "LLM_ONLY",
            "status": "PASS",
            "answer": answer,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "generation_time_sec": round(elapsed_sec, 2),
            "generation_config": GENERAL_GENERATION_CONFIG,
            "error": None,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        }

        print(
            f"  PASS | "
            f"Input: {input_tokens:,} | "
            f"Output: {output_tokens:,} | "
            f"Time: {elapsed_sec:.2f} sec"
        )

    except Exception as exc:
        elapsed_sec = time.time() - start_time

        # 2. ERROR RECOVERY CLEANUP
        flush_vram(delay_sec=0.5)

        record = {
            "question_id": question_id,
            "category": category,
            "question": question,
            "expected_points": item.get("expected_points", []),
            "model_id": MODEL_NAME,
            "mode": "LLM_ONLY",
            "status": "FAIL",
            "answer": None,
            "input_tokens": None,
            "output_tokens": None,
            "generation_time_sec": round(elapsed_sec, 2),
            "generation_config": GENERAL_GENERATION_CONFIG,
            "error": f"{type(exc).__name__}: {exc}",
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        }

        print(f"  FAIL | {type(exc).__name__}: {exc}")

    track1_general_results.append(record)

    # Incremental Checkpoint Save to disk
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(track1_general_results, f, indent=2, ensure_ascii=False)

    # 3. POST-QUERY CLEANUP
    flush_vram(delay_sec=0.2)


# =============================================================================
# SUMMARY & TELEMETRY
# =============================================================================

track1_elapsed = time.time() - track1_start
track1_pass = sum(r["status"] == "PASS" for r in track1_general_results)
track1_fail = sum(r["status"] == "FAIL" for r in track1_general_results)

# Create structured payload wrapper with top-level metadata
final_payload = {
    "metadata": {
        "track": "Track 1 - General LLM Only Baseline",
        "model_id": MODEL_NAME,
        "mode": "LLM_ONLY",
        "total_questions": len(benchmark_questions),
        "captured": len(track1_general_results),
        "pass_count": track1_pass,
        "fail_count": track1_fail,
        "total_runtime_minutes": round(track1_elapsed / 60, 2),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    },
    "results": track1_general_results
}

# Final metadata-wrapped save to disk
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(final_payload, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 90)
print("BASELINE — TRACK 1 BENCHMARK COMPLETE")
print("=" * 90)

print(f"Expected questions : {len(benchmark_questions)}")
print(f"Captured           : {len(track1_general_results)}")
print(f"PASS               : {track1_pass}")
print(f"FAIL               : {track1_fail}")
print(f"Runtime            : {track1_elapsed / 60:.2f} min")
print(f"Saved payload to   : {OUTPUT_FILE}")
print("=" * 90)

BASELINE — TRACK 1 | GENERAL LLM ONLY (NO RAG)
DIRECT INFERENCE
Model ID        : farbodtavakkoli/OTel-2.0-LLM-31B-IT
Questions       : 20

[01/20] Q01 | 5G Core
  PASS | Input: 29 | Output: 825 | Time: 161.43 sec

[02/20] Q02 | 5G Core
  PASS | Input: 30 | Output: 1,024 | Time: 236.52 sec

[03/20] Q03 | 5G RAN
  PASS | Input: 28 | Output: 967 | Time: 212.55 sec

[04/20] Q04 | 5G RAN
  PASS | Input: 28 | Output: 1,024 | Time: 235.88 sec

[05/20] Q05 | 5G SA Procedures
  PASS | Input: 28 | Output: 1,024 | Time: 235.96 sec

[06/20] Q06 | 5G SA Procedures
  PASS | Input: 28 | Output: 1,024 | Time: 236.27 sec

[07/20] Q07 | Open RAN
  PASS | Input: 34 | Output: 1,024 | Time: 237.89 sec

[08/20] Q08 | Open RAN
  PASS | Input: 37 | Output: 819 | Time: 161.75 sec

[09/20] Q09 | Cloud-Native Telecom
  PASS | Input: 30 | Output: 1,024 | Time: 236.38 sec

[10/20] Q10 | Cloud-Native Telecom
  PASS | Input: 24 | Output: 979 | Time: 216.21 sec

[11/20] Q11 | Applied Telecom Engineering
  PASS | Inp

### **Track 2 Inference**

In [17]:
import gc
import json
import time
from datetime import datetime, timezone
import torch

# =============================================================================
# HELPER: AGGRESSIVE HARDWARE SYNCHRONIZED MEMORY CLEARING
# =============================================================================

def flush_vram(delay_sec: float = 0.5):
    """
    Forces garbage collection, releases cached CUDA VRAM back to the GPU,
    synchronizes CPU/GPU threads, and introduces a brief settlement delay.
    """
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.synchronize()  # Block CPU until GPU cache release finishes
    if delay_sec > 0:
        time.sleep(delay_sec)


# =============================================================================
# CONFIGURATION & METADATA SETUP
# =============================================================================

SYSTEM_PROMPT = "You are an expert telecommunications network engineer."

# Resolve target model & tokenizer safely (supporting both otel and general variables)
active_model = globals().get("otel_model", globals().get("general_model"))
active_tokenizer = globals().get("otel_tokenizer", globals().get("general_tokenizer"))

if active_model is None or active_tokenizer is None:
    raise NameError("Neither 'otel_model'/'otel_tokenizer' nor 'general_model'/'general_tokenizer' was found in memory.")

MODEL_NAME = getattr(active_model.config, "_name_or_path", "farbodtavakkoli/OTel-2.0-LLM-31B-IT")
TIMESTAMP_STR = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
OUTPUT_FILE = f"track2_general_llm_results_{TIMESTAMP_STR}.json"

# Ensure dataset exists
if "benchmark_questions_track2" not in globals():
    raise NameError("Variable 'benchmark_questions_track2' is missing. Load your Track 2 dataset first.")

track2_general_results = []
track2_start = time.time()

# Initial hardware memory flush
flush_vram(delay_sec=1.0)


# =============================================================================
# HEADER
# =============================================================================

print("=" * 90)
print("BASELINE — TRACK 2 | GENERAL LLM ONLY (NO RAG)")
print("DIRECT INFERENCE")
print("=" * 90)
print(f"Model ID        : {MODEL_NAME}")
print(f"Questions       : {len(benchmark_questions_track2)}")
print("=" * 90)


# =============================================================================
# EXECUTE BENCHMARK
# =============================================================================

for index, item in enumerate(benchmark_questions_track2, start=1):

    # 1. PRE-QUERY CLEANUP & SYNCHRONIZATION
    flush_vram(delay_sec=0.5)

    question_id = item["evaluation_id"]
    benchmark = item["benchmark"]
    question = item["question"]
    choices = item.get("choices")

    # Format model query without target answer leakage
    if choices:
        choices_text = "\n".join(str(choice) for choice in choices)
        model_query = f"{question}\n\nChoices:\n{choices_text}"
    else:
        model_query = question

    print(f"\n[{index:02d}/{len(benchmark_questions_track2):02d}] {question_id} | {benchmark}")
    start_time = time.time()

    # =========================================================================
    # INFERENCE (wrapped in torch.inference_mode to disable autograd tracking)
    # =========================================================================
    try:
        with torch.inference_mode():
            # Generate standalone LLM response using active model/tokenizer
            answer = generate_response(
                model=active_model,
                tokenizer=active_tokenizer,
                question=model_query,
                system_prompt=SYSTEM_PROMPT,
                max_new_tokens=GENERAL_GENERATION_CONFIG.get("max_new_tokens", 500),
                do_sample=GENERAL_GENERATION_CONFIG.get("do_sample", False),
                temperature=GENERAL_GENERATION_CONFIG.get("temperature", 0.01),
                repetition_penalty=GENERAL_GENERATION_CONFIG.get("repetition_penalty", 1.1),
            )

            # Compute token telemetry using active tokenizer
            prompt_str = f"{SYSTEM_PROMPT}\n{model_query}"
            input_tokens = len(active_tokenizer.encode(prompt_str))
            output_tokens = len(active_tokenizer.encode(answer))

        elapsed_sec = time.time() - start_time

        record = {
            "question_id": question_id,
            "benchmark": benchmark,
            "question": question,
            "choices": choices,
            "expected_answer": item.get("answer"),
            "explanation": item.get("explanation"),
            "candidate_selection_score": item.get("candidate_selection_score"),
            "model_id": MODEL_NAME,
            "mode": "LLM_ONLY",
            "status": "PASS",
            "answer": answer,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "generation_time_sec": round(elapsed_sec, 2),
            "generation_config": GENERAL_GENERATION_CONFIG,
            "error": None,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        }

        print(
            f"  PASS | "
            f"Input: {input_tokens:,} | "
            f"Output: {output_tokens:,} | "
            f"Time: {elapsed_sec:.2f} sec"
        )

    except Exception as exc:
        elapsed_sec = time.time() - start_time

        # 2. ERROR RECOVERY CLEANUP
        flush_vram(delay_sec=0.5)

        record = {
            "question_id": question_id,
            "benchmark": benchmark,
            "question": question,
            "choices": choices,
            "expected_answer": item.get("answer"),
            "explanation": item.get("explanation"),
            "candidate_selection_score": item.get("candidate_selection_score"),
            "model_id": MODEL_NAME,
            "mode": "LLM_ONLY",
            "status": "FAIL",
            "answer": None,
            "input_tokens": None,
            "output_tokens": None,
            "generation_time_sec": round(elapsed_sec, 2),
            "generation_config": GENERAL_GENERATION_CONFIG,
            "error": f"{type(exc).__name__}: {exc}",
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        }

        print(f"  FAIL | {type(exc).__name__}: {exc}")

    track2_general_results.append(record)

    # Incremental Checkpoint Save to disk
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(track2_general_results, f, indent=2, ensure_ascii=False)

    # 3. POST-QUERY CLEANUP
    flush_vram(delay_sec=0.2)


# =============================================================================
# SUMMARY & TELEMETRY
# =============================================================================

track2_elapsed = time.time() - track2_start
track2_pass = sum(r["status"] == "PASS" for r in track2_general_results)
track2_fail = sum(r["status"] == "FAIL" for r in track2_general_results)

# Create structured payload wrapper with top-level metadata
final_payload = {
    "metadata": {
        "track": "Track 2 - General LLM Only Baseline",
        "model_id": MODEL_NAME,
        "mode": "LLM_ONLY",
        "total_questions": len(benchmark_questions_track2),
        "captured": len(track2_general_results),
        "pass_count": track2_pass,
        "fail_count": track2_fail,
        "total_runtime_minutes": round(track2_elapsed / 60, 2),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    },
    "results": track2_general_results
}

# Final metadata-wrapped save to disk
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(final_payload, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 90)
print("BASELINE — TRACK 2 BENCHMARK COMPLETE")
print("=" * 90)

print(f"Expected questions : {len(benchmark_questions_track2)}")
print(f"Captured           : {len(track2_general_results)}")
print(f"PASS               : {track2_pass}")
print(f"FAIL               : {track2_fail}")
print(f"Runtime            : {track2_elapsed / 60:.2f} min")
print(f"Artifacts saved    : {OUTPUT_FILE}")
print("=" * 90)

BASELINE — TRACK 2 | GENERAL LLM ONLY (NO RAG)
DIRECT INFERENCE
Model ID        : farbodtavakkoli/OTel-2.0-LLM-31B-IT
Questions       : 32

[01/32] T2-01 | 3gpp_tsg
  PASS | Input: 1,307 | Output: 13 | Time: 7.48 sec

[02/32] T2-02 | 3gpp_tsg
  PASS | Input: 1,577 | Output: 8 | Time: 5.72 sec

[03/32] T2-03 | 3gpp_tsg
  PASS | Input: 1,365 | Output: 8 | Time: 4.85 sec

[04/32] T2-04 | 3gpp_tsg
  PASS | Input: 1,240 | Output: 8 | Time: 4.25 sec

[05/32] T2-05 | oranbench
  PASS | Input: 89 | Output: 371 | Time: 52.59 sec

[06/32] T2-06 | oranbench
  PASS | Input: 123 | Output: 247 | Time: 31.82 sec

[07/32] T2-07 | oranbench
  PASS | Input: 71 | Output: 227 | Time: 26.68 sec

[08/32] T2-08 | oranbench
  PASS | Input: 78 | Output: 174 | Time: 19.22 sec

[09/32] T2-09 | sixg_bench
  PASS | Input: 489 | Output: 325 | Time: 83.94 sec

[10/32] T2-10 | sixg_bench
  PASS | Input: 557 | Output: 456 | Time: 140.40 sec

[11/32] T2-11 | sixg_bench
  PASS | Input: 496 | Output: 286 | Time: 72.23 se